# MIDOG++ count-error region selection — Colab runner

Runs the pipeline defined in this repository on a CUDA runtime. The local machine (Windows,
no GPU) is used for audit, tests and the interface only; see `docs/COMPUTE.md`.

**Before running:** this notebook expects the project code in your Google Drive at
`MyDrive/midogcx_project` and specimen images at `MyDrive/midogpp_data`.
The 55-specimen cohort is 7.52 GiB.

This notebook produces real experimental output. It does not fabricate or approximate any
number: if a step cannot run, it stops and says why.

In [ ]:
import subprocess, sys, os, shutil, textwrap
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)
print("python", sys.version.split()[0])

In [ ]:
# Colab ships CUDA torch already; only add what is missing.
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "opencv-python-headless", "PyYAML", "scikit-learn"], check=True)
import torch, torchvision
print("torch", torch.__version__, "cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Switch the runtime to a GPU: Runtime > Change runtime type"

## 1. Project code and paths

If you have not uploaded the project yet, upload the folder as a zip to MyDrive, then set
`PROJECT_DIR` accordingly. The notebook never edits the project; it only runs its scripts.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/midogcx_project"
DATA_ROOT   = "/content/drive/MyDrive/midogpp_data"   # 7.52 GiB for the cohort
os.environ["MIDOG_DATA_ROOT"] = DATA_ROOT
os.makedirs(DATA_ROOT, exist_ok=True)

assert os.path.isdir(PROJECT_DIR), f"project not found at {PROJECT_DIR}"
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print("cwd:", os.getcwd())
print("free space (GB):", round(shutil.disk_usage("/content").free / 1e9, 1))

In [ ]:
# Sanity: code must pass its own tests before any training time is spent.
!python -m pytest tests/ -q

## 2. Dataset, frozen cohort and frozen region grid

Metadata ships with the project. Both frozen artefacts are **verified, never
regenerated**: the split and the region grid define every selection, budget and
previously reported number, so a remote runtime must not be able to rewrite them.


In [ ]:
# Verifies the frozen cohort instead of rewriting it: the split is immutable.
!python scripts/build_cohort.py --check


In [ ]:
# Exact download sizes first. Nothing is fetched by --dry-run.
!python scripts/download_specimens.py --dry-run

In [ ]:
# The cohort is 7.52 GiB. The downloader refuses to start if the disk cannot hold it,
# resumes partial files, and verifies byte-exact sizes.
!python scripts/download_specimens.py

## 3. Regions and normalisation statistics

Reads every cohort specimen once: writes the non-overlapping 512x512 region grid, the
physical spacing table, and channel statistics computed **only** from source training
specimens.

In [ ]:
# Verifies the frozen region grid (regions.csv, geometry.csv, normalization.json)
# against a fresh recomputation and writes nothing: selections and budgets are
# defined against this exact grid.
!python scripts/prepare_regions.py --check


## 4. Source-domain detector

A real run needs far more than the smoke-test budget. Measure first: the script logs
seconds per update, so a session can be cut short without losing the result. The checkpoint
stores its own architecture, normalisation and update count.

In [ ]:
# Pilot-scale first run. Increase `training.updates` once throughput is known.
# One --set with several pairs: repeating --set keeps only the last flag, which
# silently dropped batch_size and device on the first Colab run.
!python scripts/train_detector.py --tag source_baseline --set training.device=cuda training.batch_size=4 training.updates=500


In [ ]:
import glob, os
ckpts = sorted(glob.glob("outputs/runs/*/checkpoint.pth"), key=os.path.getmtime)
print("\n".join(ckpts[-3:]))
SOURCE_CKPT = ckpts[-1]
print("using:", SOURCE_CKPT)


## 5. Acquisition experiment

Threshold calibration, predictor fitting on out-of-sample source specimens, the five
rankings, matched adaptation branches and independent melanoma assessment. A `source_only`
control is always evaluated.

In [ ]:
# GATE: evaluate the unadapted control before spending GPU time on any branch.
# If precision/recall/F1 are near zero here, the acquisition comparison is
# meaningless, so fix the baseline first.
!python scripts/run_acquisition_experiment.py --checkpoint {SOURCE_CKPT} --tag baseline_gate --device cuda --stop-after-control


In [ ]:
# Braces, not $VAR: IPython substitutes the Python variable before the shell sees it.
!python scripts/run_acquisition_experiment.py     --checkpoint {SOURCE_CKPT} --tag pilot --device cuda

In [ ]:
import pandas as pd
run = sorted(glob.glob("outputs/experiments/*"), key=os.path.getmtime)[-1]
print(run)
display(pd.read_csv(os.path.join(run, "comparison.csv")))
display(pd.read_json(os.path.join(run, "predictor_report.json"), typ="series"))

## 6. Second seed (optional, needed for a variability estimate)

Everything is seeded from `project.seed`; changing it produces an independent replicate.
Report seed variability separately from specimen variability.

In [ ]:
# Independent replicate (report seed variability separately from specimen variability).
# !python scripts/train_detector.py --tag source_seed2 --set project.seed=7 --set training.device=cuda
# !python scripts/run_acquisition_experiment.py --checkpoint outputs/runs/<seed2_run>/checkpoint.pth --tag pilot_seed2 --device cuda --set project.seed=7

## 7. Persist results

Small result files are what you actually need to keep; checkpoints can be regenerated.

In [ ]:
ARCHIVE = "/content/drive/MyDrive/midogcx_results"
shutil.make_archive(os.path.join(ARCHIVE, "results_" + os.path.basename(run)), "zip", run)
print("archived:", ARCHIVE)

## What to report

Use `comparison.csv` for the method table and `paired_comparisons.json` for per-specimen
differences with intervals. Always report counting error **together with** precision, recall,
F1 and the raw FP/FN counts — count error can be small while false positives and false
negatives cancel. Do not present any pilot-era number from the research blueprint as a result
of this run.